# Exploracion inicial: maestra de articulos

Notebook de arranque para revisar `ie_maestra_articulos.csv` sin versionar el archivo grande en Git.

## Objetivos

- Cargar el CSV con el separador correcto.
- Preservar codigos con ceros a la izquierda.
- Revisar dimensiones, nulos, duplicados y distribucion por sector.
- Dejar algunas celdas listas para busquedas y exploracion de texto.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [ ]:
DATA_PATH = Path("ie_maestra_articulos.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No encuentro {DATA_PATH.resolve()}. Coloca el CSV en la raiz del repo antes de ejecutar el notebook."
    )

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    encoding="utf-8",
    dtype={"idarticu": "string", "idsector": "string"},
)

df.shape

In [ ]:
df.head(10)

In [ ]:
df.info(memory_usage="deep")

## Calidad basica

In [ ]:
summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": df.isna().mean().mul(100).round(2),
        "n_unique": df.nunique(dropna=True),
    }
)

summary

In [ ]:
duplicate_id_count = df.duplicated("idarticu").sum()
duplicate_row_count = df.duplicated().sum()

duplicate_id_count, duplicate_row_count

In [ ]:
df.loc[df.duplicated("idarticu", keep=False)].sort_values("idarticu").head(20)

## Sectores

In [ ]:
sector_counts = (
    df.groupby(["idsector", "desc_sector"], dropna=False)
    .size()
    .reset_index(name="n_articulos")
    .sort_values("n_articulos", ascending=False)
)

sector_counts

In [ ]:
ax = sector_counts.head(20).sort_values("n_articulos").plot.barh(
    x="desc_sector",
    y="n_articulos",
    figsize=(10, 7),
    legend=False,
)
ax.set_title("Top 20 sectores por numero de articulos")
ax.set_xlabel("Articulos")
ax.set_ylabel("Sector")
plt.tight_layout()

## Texto de articulos

In [ ]:
df["desc_larga_articulo_len"] = df["desc_larga_articulo"].astype("string").str.len()
df["desc_larga_articulo_len"].describe()

In [ ]:
df.nlargest(20, "desc_larga_articulo_len")[["idarticu", "desc_larga_articulo", "idsector", "desc_sector", "desc_larga_articulo_len"]]

In [ ]:
def buscar_articulos(texto, n=20):
    mask = df["desc_larga_articulo"].astype("string").str.contains(texto, case=False, na=False, regex=False)
    return df.loc[mask, ["idarticu", "desc_larga_articulo", "idsector", "desc_sector"]].head(n)

buscar_articulos("CARREFOUR")

## Siguientes pasos posibles

- Normalizar descripciones de articulos para busquedas por marca/producto.
- Revisar si `idarticu` identifica una fila unica.
- Cruzar esta maestra con tickets, ventas o clientes cuando esten disponibles.